# 02. Comparative DBI Assessment

Notebook này sử dụng Digital Burnout Indicator Framework đã được xây dựng ở Notebook 01 để phát triển phương pháp đánh giá Digital Burnout trên cả bộ dữ liệu quốc tế và bộ dữ liệu sinh viên Việt Nam.

Các nhiệm vụ chính bao gồm:

- Xây dựng trọng số cho từng Digital Burnout Indicator.
- Tính điểm cho từng Indicator.
- Tổng hợp điểm theo từng DBI Dimension.
- Xây dựng Digital Burnout Index (DBI).
- So sánh kết quả đánh giá giữa hai bộ dữ liệu.
- Chuẩn bị dữ liệu phục vụ hệ thống mô phỏng Streamlit.

# 0. Set Up

Chuẩn bị môi trường làm việc cho quá trình xây dựng và đánh giá Digital Burnout Index (DBI).

In [1]:
# Import các thư viện phục vụ xử lý dữ liệu

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Thiết lập tùy chọn hiển thị của pandas

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.6f}".format)

In [3]:
# Khai báo đường dẫn tới thư mục dữ liệu

processed_path = Path("../../data/processed")

international_path = processed_path / "international_dataset"

vietnam_path = processed_path / "vietnam_dataset"

cross_dataset_path = processed_path / "cross_dataset_analysis"

In [4]:
# Kiểm tra sự tồn tại của các thư mục dữ liệu

directory_summary = pd.DataFrame({
    "Directory": [
        "International Dataset",
        "Vietnam Dataset",
        "Cross Dataset Analysis"
    ],
    "Path": [
        international_path,
        vietnam_path,
        cross_dataset_path
    ],
    "Exists": [
        international_path.exists(),
        vietnam_path.exists(),
        cross_dataset_path.exists()
    ]
})

display(directory_summary)

print("Đã hoàn thành quá trình kiểm tra cấu trúc thư mục dữ liệu.")

,Directory,Path,Exists
0,International Dataset,..\..\data\processed\international_dataset,True
1,Vietnam Dataset,..\..\data\processed\vietnam_dataset,True
2,Cross Dataset Analysis,..\..\data\processed\cross_dataset_analysis,True


Đã hoàn thành quá trình kiểm tra cấu trúc thư mục dữ liệu.


# 1. Load Data

Đọc toàn bộ dữ liệu cần thiết phục vụ quá trình xây dựng và đánh giá Digital Burnout Index (DBI).

Phần này thực hiện:

- Đọc bộ dữ liệu quốc tế sau tiền xử lý.
- Đọc bộ dữ liệu sinh viên Việt Nam sau tiền xử lý.
- Đọc Digital Burnout Indicator Framework đã xây dựng ở Notebook 01.
- Kiểm tra kích thước và cấu trúc của từng bộ dữ liệu.


In [5]:
# Đọc bộ dữ liệu quốc tế đã được làm sạch

international_df = pd.read_csv(

    international_path / "digital_burnout_cleaned.csv"

)

print("Đã tải bộ dữ liệu quốc tế thành công.")

print(f"Số lượng quan sát: {international_df.shape[0]:,}")

print(f"Số lượng biến: {international_df.shape[1]}")

Đã tải bộ dữ liệu quốc tế thành công.
Số lượng quan sát: 5,000,000
Số lượng biến: 35


In [6]:
# Đọc bộ dữ liệu sinh viên Việt Nam đã được làm sạch

vietnam_df = pd.read_csv(

    vietnam_path / "vn_digital_burnout_cleaned.csv"

)

print("Đã tải bộ dữ liệu sinh viên Việt Nam thành công.")

print(f"Số lượng quan sát: {vietnam_df.shape[0]:,}")

print(f"Số lượng biến: {vietnam_df.shape[1]}")

Đã tải bộ dữ liệu sinh viên Việt Nam thành công.
Số lượng quan sát: 590
Số lượng biến: 23


In [7]:
# Đọc Digital Burnout Indicator Framework

dbi_framework_df = pd.read_csv(

    cross_dataset_path / "dbi_framework.csv"

)

print("Đã tải Digital Burnout Indicator Framework thành công.")

print(f"Số lượng indicator: {dbi_framework_df.shape[0]}")

Đã tải Digital Burnout Indicator Framework thành công.
Số lượng indicator: 12


In [8]:
# Tổng hợp thông tin ba bộ dữ liệu

dataset_overview = pd.DataFrame({

    "Dataset":[

        "International Dataset",

        "Vietnam Dataset",

        "Digital Burnout Framework"

    ],

    "Observations":[

        international_df.shape[0],

        vietnam_df.shape[0],

        dbi_framework_df.shape[0]

    ],

    "Variables":[

        international_df.shape[1],

        vietnam_df.shape[1],

        dbi_framework_df.shape[1]

    ]

})

display(dataset_overview)

,Dataset,Observations,Variables
0,International Dataset,5000000,35
1,Vietnam Dataset,590,23
2,Digital Burnout Framework,12,13


# 2. Prepare Comparative Dataset

Chuẩn bị bộ dữ liệu so sánh Digital Burnout giữa bộ dữ liệu quốc tế và dữ liệu khảo sát sinh viên Việt Nam.

Do hai bộ dữ liệu có nguồn gốc và cấu trúc khác nhau, quá trình này tập trung vào việc ánh xạ các đặc trưng ban đầu về cùng một hệ thống Indicator trong DBI Framework.

Quy trình chuẩn bị dữ liệu bao gồm:

1. Chuẩn hóa mapping giữa Feature và Indicator.
2. Xây dựng bộ dữ liệu Indicator Score cho International Dataset.
3. Chuẩn hóa cấu trúc dữ liệu Vietnam DBI Assessment.
4. Tạo bộ dữ liệu chung phục vụ phân tích so sánh.

In [17]:
# Đọc các file Framework DBI đã xây dựng từ Notebook 01

import pandas as pd


dbi_framework_path = (
    "../../data/processed/dbi_framework/"
    "dbi_framework.csv"
)


feature_mapping_path = (
    "../../data/processed/dbi_framework/"
    "feature_indicator_mapping.csv"
)


indicator_weights_path = (
    "../../data/processed/dbi_framework/"
    "indicator_weights.csv"
)


dbi_framework = pd.read_csv(
    dbi_framework_path
)


feature_mapping = pd.read_csv(
    feature_mapping_path
)


indicator_weights = pd.read_csv(
    indicator_weights_path
)


print("Đã tải Framework DBI thành công.")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/processed/dbi_framework/feature_indicator_mapping.csv'